# SoninkeGPT — audios des nouveaux candidats lexicaux avec la voix du référent

Ce notebook reprend le protocole du lot tête et visage et prépare la génération audio des candidats CNRS/LGIDF et Peace Corps.

- La file d'attente comporte 336 candidats, tous encore à vérifier lexicalement.
- La génération ne vaut ni validation de la traduction ni validation de la prononciation.
- Le notebook s'arrête si aucun GPU n'est disponible ou si le modèle ne confirme pas explicitement la prise en charge du soninké.
- Les fichiers existants sont réutilisés; limite par session pour éviter des centaines de générations d'un coup.
- Les poids OmniVoice sont documentés avec une licence non commerciale : vérifier les droits avant tout usage commercial.

In [ ]:
%pip -q install omnivoice soundfile pandas

In [ ]:
from pathlib import Path
from google.colab import drive, files
from IPython.display import Audio, display
import pandas as pd
import numpy as np
import soundfile as sf
import hashlib, json, zipfile, datetime, urllib.request

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/SONINKE_TTS_LIBRE')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_DIR = PROJECT_DIR / 'new_lexicon_candidates_batch'
AUDIO_DIR = BATCH_DIR / 'audio'
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# Priorité à la référence phrase validée; sinon, construire une référence à partir des 20 WAV fournis.
REF_PATH = PROJECT_DIR / 'reference_soninke.wav'
REF_TEXT = 'Daga goli. Li taxu.'
HEAD_FACE_DIR = PROJECT_DIR / 'head_face_batch'
HEAD_FACE_AUDIO_DIR = HEAD_FACE_DIR / 'audio'
HEAD_FACE_MANIFEST = HEAD_FACE_DIR / 'manifest.csv'
REVIEW_ZIP = HEAD_FACE_DIR / 'head-face-audio-review.zip'

if not REF_PATH.exists():
    wavs = list(HEAD_FACE_AUDIO_DIR.glob('SNK-LEX-*.wav')) if HEAD_FACE_AUDIO_DIR.exists() else []
    if len(wavs) < 10 and REVIEW_ZIP.exists():
        with zipfile.ZipFile(REVIEW_ZIP, 'r') as zf:
            zf.extractall(HEAD_FACE_DIR)
        wavs = list(HEAD_FACE_AUDIO_DIR.glob('SNK-LEX-*.wav'))
    if len(wavs) < 10:
        print('Je vais utiliser le ZIP des 20 audios déjà produits comme référence vocale.')
        print('Téléverse head-face-audio-review.zip depuis ton appareil.')
        uploaded = files.upload()
        for name, blob in uploaded.items():
            target = HEAD_FACE_DIR / name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(blob)
            if name.lower().endswith('.zip'):
                with zipfile.ZipFile(target, 'r') as zf:
                    zf.extractall(HEAD_FACE_DIR)
        wavs = list(HEAD_FACE_AUDIO_DIR.glob('SNK-LEX-*.wav'))
    if len(wavs) < 10 or not HEAD_FACE_MANIFEST.exists():
        raise FileNotFoundError('Impossible de construire la référence : le ZIP doit contenir manifest.csv et les WAV SNK-LEX-*.wav.')
    manifest = pd.read_csv(HEAD_FACE_MANIFEST, dtype=str).fillna('')
    parts, text_parts, sr_ref = [], [], None
    for _, r in manifest.iterrows():
        wav_path = HEAD_FACE_AUDIO_DIR / f"{r['id']}.wav"
        if not wav_path.exists() or not str(r.get('text', '')).strip():
            continue
        samples, sr = sf.read(wav_path)
        if samples.ndim > 1:
            samples = samples.mean(axis=1)
        if sr_ref is None:
            sr_ref = sr
        if sr != sr_ref:
            raise ValueError('Les 20 audios ne partagent pas la même fréquence d’échantillonnage.')
        parts.append(samples.astype(np.float32))
        text_parts.append(str(r['text']).strip())
    if len(parts) < 10:
        raise ValueError('Pas assez de WAV valides pour fabriquer la référence vocale.')
    silence = np.zeros(int(0.25 * sr_ref), dtype=np.float32)
    combined = []
    for i, part in enumerate(parts):
        if i:
            combined.append(silence)
        combined.append(part)
    REF_PATH = PROJECT_DIR / 'reference_from_20_head_face_words.wav'
    sf.write(REF_PATH, np.concatenate(combined), sr_ref)
    REF_TEXT = ' '.join(text_parts)
    print('Référence construite à partir de', len(parts), 'audios du lot tête-visage.')
print('Référence de voix :', REF_PATH)
print('Transcription associée :', REF_TEXT)
display(Audio(str(REF_PATH)))

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('ARRÊT SÛR : aucun GPU disponible. Ne paie pas pour continuer; réessaie avec une session gratuite plus tard.')
print('GPU disponible :', torch.cuda.get_device_name(0))
from omnivoice import OmniVoice
model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map='cuda:0', dtype=torch.float16)
lang_names = [str(x).lower() for x in model.supported_language_names()]
if not any('sonink' in x for x in lang_names):
    raise RuntimeError('ARRÊT : cette version du modèle ne confirme pas la langue Soninke.')
SAMPLE_RATE = int(model.sampling_rate)
print('Fréquence du modèle :', SAMPLE_RATE)

In [ ]:
CSV_URL = 'https://raw.githubusercontent.com/aydiarra-star/soninkeGPT/research/peace-corps-soninke-audio-lexicon/data/audio/new-lexicon-candidates-production-queue.csv'
queue_path = BATCH_DIR / 'new-lexicon-candidates-production-queue.csv'
with urllib.request.urlopen(CSV_URL, timeout=30) as response:
    queue_path.write_bytes(response.read())
df = pd.read_csv(queue_path, dtype=str).fillna('')
required = {'audio_id', 'soninke_source_form', 'french_candidate', 'source_name', 'source_url'}
if not required.issubset(df.columns):
    raise ValueError(f'Colonnes manquantes : {required - set(df.columns)}')
df = df.rename(columns={'audio_id': 'id', 'soninke_source_form': 'text', 'french_candidate': 'francais', 'source_name': 'source', 'source_url': 'attribution'})
df['license'] = 'candidat lexical; vérifier les droits de la source et du modèle'
if df['id'].duplicated().any():
    raise ValueError('Les identifiants du lot doivent être uniques.')
print('Candidats dans la file :', len(df))
print('Chaque session crée au maximum 20 nouveaux WAV; relancer pour continuer.')
display(df[['id', 'text', 'francais', 'source']].head(30))

## Génération des WAV par petits lots

Le manifeste est sauvegardé après chaque entrée. Le notebook réutilise les WAV déjà présents et ne génère que jusqu'à 20 nouveaux fichiers par exécution. Relancer la cellule de génération pour poursuivre le lot. Les formes et traductions restent des candidates à vérifier.

In [ ]:
MANIFEST = BATCH_DIR / 'manifest.csv'
MAX_NEW_AUDIO_PER_RUN = 20
existing = {}
if MANIFEST.exists():
    old = pd.read_csv(MANIFEST, dtype=str).fillna('')
    existing = {r['id']: r for _, r in old.iterrows()}

records = []
newly_generated = 0
for _, row in df.iterrows():
    item_id, phrase = row['id'].strip(), row['text'].strip()
    wav_path = AUDIO_DIR / f'{item_id}.wav'
    previous = existing.get(item_id, {})
    error = ''
    if wav_path.exists() and wav_path.stat().st_size > 1000:
        status = 'audio_present_reused'
    else:
        if newly_generated >= MAX_NEW_AUDIO_PER_RUN:
            break
        status = 'generation_error'
        try:
            print(f"Génération : {item_id} — {phrase}")
            generated = model.generate(text=phrase, language='soninke', ref_audio=str(REF_PATH), ref_text=REF_TEXT, num_step=32)
            if generated is None or len(generated) == 0 or len(generated[0]) == 0:
                raise RuntimeError('Le modèle n’a retourné aucun échantillon audio.')
            samples = np.asarray(generated[0]).squeeze()
            if samples.ndim != 1 or not np.isfinite(samples).all():
                raise RuntimeError('Audio invalide : forme inattendue ou valeurs non finies.')
            sf.write(str(wav_path), samples, SAMPLE_RATE, subtype='PCM_16')
            status = 'generated_pending_human_review'
            newly_generated += 1
            display(Audio(str(wav_path)))
        except Exception as exc:
            error = f'{type(exc).__name__}: {exc}'
            print('ERREUR — arrêt du lot pour éviter des répétitions coûteuses :', error)
    duration, peak, sha256 = '', '', ''
    if wav_path.exists() and wav_path.stat().st_size > 1000:
        info = sf.info(str(wav_path))
        audio_data, _ = sf.read(str(wav_path), always_2d=True)
        duration = round(float(info.duration), 3)
        peak = round(float(np.max(np.abs(audio_data))) if audio_data.size else 0.0, 5)
        sha256 = hashlib.sha256(wav_path.read_bytes()).hexdigest()
    records.append({'id': item_id, 'text': phrase, 'francais': row['francais'], 'source': row['source'], 'license': row['license'], 'attribution': row['attribution'], 'audio_file': str(wav_path) if wav_path.exists() else '', 'technical_status': status, 'duration_seconds': duration, 'peak_amplitude': peak, 'sha256': sha256, 'generation_error': error, 'human_pronunciation_validation': previous.get('human_pronunciation_validation', 'pending'), 'human_notes': previous.get('human_notes', '')})
    updated = pd.DataFrame(records)
    if MANIFEST.exists():
        old_df = pd.read_csv(MANIFEST, dtype=str).fillna('')
        old_df = old_df[~old_df['id'].isin(updated['id'])]
        updated = pd.concat([old_df, updated], ignore_index=True)
    updated.to_csv(MANIFEST, index=False, encoding='utf-8-sig')
    if error:
        break

print('Manifeste :', MANIFEST)
display(pd.DataFrame(records)[['id','text','technical_status','duration_seconds','human_pronunciation_validation']])
print('Nouveaux WAV créés pendant cette session :', newly_generated)
print('Les fichiers générés restent à écouter et à valider par le référent.')

In [ ]:
zip_path = BATCH_DIR / 'new-lexicon-candidates-audio-review.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    if MANIFEST.exists():
        zf.write(MANIFEST, arcname='manifest.csv')
    for wav_path in sorted(AUDIO_DIR.glob('*.wav')):
        zf.write(wav_path, arcname=f'audio/{wav_path.name}')
print('Archive de contrôle :', zip_path)
print('Les WAV restent dans Google Drive. Écouter les résultats et confirmer la prononciation avant toute intégration au dictionnaire.')

## Après la génération

1. Relancer la cellule de génération pour poursuivre les candidats par lots de 20 nouveaux WAV.
2. Vérifier chaque audio et la traduction associée avec le référent linguistique.
3. Si le GPU ou la prise en charge de Soninké bloque, conserver le message d'erreur et ne pas déclarer le lot réussi.
4. Transmettre new-lexicon-candidates-audio-review.zip pour l'audit des WAV et du manifeste.
5. Les candidats ne doivent pas être promus dans le lexique de production avant vérification linguistique.